# Newton Method

SPD methods solve f(x)=0.5*x.T@A@x-b.T@x. A small residual does not guarantee a small solution error. Stationary methods may diverge on an SPD matrix; inspect converged. Newton uses an exact Hessian solve, reusing a sparse LU or dense Cholesky factorization. Factorization time is included.

Run from the repository folder or any folder below it. Requires NumPy, SciPy and Numba.


In [1]:
import time
import sys
from pathlib import Path
import numpy as np
from scipy.sparse import csr_matrix, issparse

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside Numerical_Linear_Algebra_Algorithms")
sys.path.insert(0, str(ROOT))
from common import loader, data_path, lu_solve, relative_residual
from Iterative_Methods.newton import newton


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


## 3×3


In [2]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = newton(A, b)
result, seconds = timed(newton, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
assert relative_residual(A, x, b) < 1e-8


Seconds: 0.0008620840089861304
{'iterations': 1, 'residual': 1.1563098121742844e-16, 'converged': True}
Relative residual: 1.1563098121742844e-16
Relative solution error: 1.812986607347358e-16


## SuiteSparse

Change MATRIX to an existing file. bcsstk09 is 1,083×1,083; bcsstk17 is 10,974×10,974; west1505 is nonsymmetric and requires pivoting. No 10^5 matrix exists in this repo. Keep CSR storage for larger tests.


In [3]:
MATRIX = "bcsstk09"
MAT_PATH = None
A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=False)
print("Matrix:", MATRIX, "shape:", A.shape)
x_true = np.ones(A.shape[0])
b = A @ x_true


Matrix: bcsstk09 shape: (1083, 1083)


In [4]:
result, seconds = timed(newton, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))


Seconds: 0.0067740420054178685
{'iterations': 1, 'residual': 7.722548142601909e-16, 'converged': True}
Relative residual: 7.722548142601909e-16
Relative solution error: 2.4262438939068703e-13
